# 🔧 Mantenimiento Predictivo de Motores con IA
**Dataset:** AI4I 2020 Predictive Maintenance (UCI Repository)  
**Objetivo:** Clasificar si un motor fallará o no basándose en variables de sensores  
**Herramientas:** Python, scikit-learn, XGBoost, matplotlib

---

## 📦 Paso 1: Instalación e importación de librerías

In [ ]:
# Instalar librerías necesarias (solo si estás en Colab)
!pip install xgboost --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas correctamente')

## 📥 Paso 2: Carga del Dataset

Usamos el dataset **AI4I 2020** directamente desde UCI (sin necesidad de descargar nada manualmente).

In [ ]:
# Cargar dataset directamente desde UCI Repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv'
df = pd.read_csv(url)

print('✅ Dataset cargado exitosamente')
print(f'   Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
df.head()

## 🔍 Paso 3: Exploración de Datos (EDA)

In [ ]:
print('--- Información general ---')
print(df.info())
print('\n--- Estadísticas descriptivas ---')
df.describe()

In [ ]:
print('--- Valores nulos por columna ---')
print(df.isnull().sum())

print('\n--- Distribución de la variable objetivo (Machine failure) ---')
print(df['Machine failure'].value_counts())
print(f'\nPorcentaje de fallas: {df["Machine failure"].mean()*100:.2f}%')

In [ ]:
# Visualización 1: Distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras de fallas
counts = df['Machine failure'].value_counts()
axes[0].bar(['Normal (0)', 'Falla (1)'], counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.5)
axes[0].set_title('Distribución de Fallas en Motores', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cantidad de registros')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Distribución de temperatura del aire
df.groupby('Machine failure')['Air temperature [K]'].plot(
    kind='hist', ax=axes[1], alpha=0.7, bins=30,
    color=['#2ecc71', '#e74c3c'], legend=True
)
axes[1].set_title('Temperatura del Aire por Estado del Motor', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Temperatura del Aire [K]')
axes[1].legend(['Normal', 'Falla'])

plt.tight_layout()
plt.savefig('distribucion_datos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada como distribucion_datos.png')

In [ ]:
# Visualización 2: Correlación entre variables numéricas
features_num = ['Air temperature [K]', 'Process temperature [K]',
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']

plt.figure(figsize=(10, 7))
corr = df[features_num].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Mapa de Correlación de Variables del Motor', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada como correlacion.png')

## ⚙️ Paso 4: Preprocesamiento de Datos

In [ ]:
# Eliminar columnas no relevantes para el modelo
df_model = df.drop(columns=['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])

# Codificar la columna 'Type' (L, M, H) a valores numéricos
le = LabelEncoder()
df_model['Type'] = le.fit_transform(df_model['Type'])
print(f'Tipos codificados: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Separar features (X) y variable objetivo (y)
X = df_model.drop(columns=['Machine failure'])
y = df_model['Machine failure']

print(f'\n✅ Features de entrenamiento: {list(X.columns)}')
print(f'   Shape X: {X.shape} | Shape y: {y.shape}')

In [ ]:
# División train/test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalización con StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'✅ Datos divididos:')
print(f'   Entrenamiento: {X_train.shape[0]} muestras')
print(f'   Prueba:        {X_test.shape[0]} muestras')

## 🤖 Paso 5: Entrenamiento de Modelos

Comparamos dos modelos: **Random Forest** y **XGBoost**

In [ ]:
# --- MODELO 1: Random Forest ---
print('🌲 Entrenando Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'  # Compensa el desbalance de clases
)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

print('\n--- Resultados Random Forest ---')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_rf)*100:.2f}%')
print('\nReporte completo:')
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Falla']))

In [ ]:
# --- MODELO 2: XGBoost ---
print('⚡ Entrenando XGBoost...')
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()  # Para desbalance

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)

print('\n--- Resultados XGBoost ---')
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_xgb)*100:.2f}%')
print('\nReporte completo:')
print(classification_report(y_test, y_pred_xgb, target_names=['Normal', 'Falla']))

## 📊 Paso 6: Visualización de Resultados

In [ ]:
# Matrices de confusión comparadas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title, color in zip(
    axes,
    [y_pred_rf, y_pred_xgb],
    ['Random Forest', 'XGBoost'],
    ['Blues', 'Oranges']
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Falla'])
    disp.plot(ax=ax, colorbar=False, cmap=color)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    ax.set_title(f'{title}\nAccuracy: {acc*100:.1f}% | F1: {f1*100:.1f}%',
                 fontsize=13, fontweight='bold')

plt.suptitle('Comparación de Modelos — Mantenimiento Predictivo de Motores',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada como matrices_confusion.png')

In [ ]:
# Importancia de variables (Random Forest)
feat_names = X.columns.tolist()
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
colors = ['#e74c3c' if importances[i] > np.mean(importances) else '#3498db' for i in indices]
bars = plt.bar(range(len(feat_names)), importances[indices], color=colors, edgecolor='black', width=0.6)
plt.xticks(range(len(feat_names)), [feat_names[i] for i in indices], rotation=25, ha='right')
plt.title('Importancia de Variables — Random Forest', fontsize=14, fontweight='bold')
plt.ylabel('Importancia relativa')
plt.axhline(y=np.mean(importances), color='gray', linestyle='--', alpha=0.7, label='Promedio')
plt.legend()

# Etiquetas de valor sobre cada barra
for bar, imp in zip(bars, importances[indices]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{imp:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('importancia_variables.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada como importancia_variables.png')

## 🔮 Paso 7: Predicción con Nuevos Datos

Simula cómo el modelo predice en tiempo real dado un nuevo registro de sensores.

In [ ]:
# Ejemplo: ingresa los valores de un motor nuevo
nuevo_motor = pd.DataFrame([{
    'Type': 2,                          # 0=H, 1=L, 2=M
    'Air temperature [K]': 298.5,       # Temperatura del aire en Kelvin
    'Process temperature [K]': 309.0,   # Temperatura del proceso en Kelvin
    'Rotational speed [rpm]': 1500,     # Velocidad rotacional
    'Torque [Nm]': 42.0,               # Torque
    'Tool wear [min]': 180              # Desgaste de herramienta en minutos
}])

# Escalar y predecir
nuevo_escalado = scaler.transform(nuevo_motor)
pred_rf = rf_model.predict(nuevo_escalado)[0]
prob_rf = rf_model.predict_proba(nuevo_escalado)[0]
pred_xgb = xgb_model.predict(nuevo_escalado)[0]
prob_xgb = xgb_model.predict_proba(nuevo_escalado)[0]

estado = lambda p: '🔴 FALLA DETECTADA' if p == 1 else '🟢 NORMAL'

print('='*50)
print('        DIAGNÓSTICO DEL MOTOR')
print('='*50)
print(f'Random Forest:  {estado(pred_rf)}')
print(f'  Probabilidad de falla: {prob_rf[1]*100:.1f}%')
print(f'\nXGBoost:        {estado(pred_xgb)}')
print(f'  Probabilidad de falla: {prob_xgb[1]*100:.1f}%')
print('='*50)

## ✅ Resumen del Proyecto

| Aspecto | Detalle |
|---|---|
| **Dataset** | AI4I 2020 Predictive Maintenance (UCI) |
| **Modelos** | Random Forest, XGBoost |
| **Variables** | Temperatura, velocidad, torque, desgaste |
| **Objetivo** | Clasificar falla / no falla en motores |
| **Herramientas** | Python, scikit-learn, XGBoost, matplotlib |

---

### 🚀 Próximos pasos opcionales para mejorar el proyecto:
1. **Optimizar hiperparámetros** con `GridSearchCV`
2. **Probar una red neuronal** con TensorFlow/Keras
3. **Crear un dashboard interactivo** con Streamlit
4. **Subir el proyecto a GitHub** con un README profesional

---
*Proyecto desarrollado con Python para currículum — Ingeniería Electrónica + IA*